In [1]:
# Colab installation
!pip install -q git+https://github.com/2forts/qcirclab_repo.git


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import numpy as np
from itertools import product

from qcirclab import Circuit, cuccaro_adder, controlled_increment


## Utility functions

The helpers below make it easier to inspect states, build small reference permutation unitaries, and verify measurement-free circuits algebraically.


In [3]:
def pretty_state(state, n_qubits=None, atol=1e-10):
    """Print nonzero amplitudes in computational-basis notation."""
    state = np.asarray(state, dtype=complex).reshape(-1)
    if n_qubits is None:
        n_qubits = int(np.log2(state.size))
    terms = []
    for i, amp in enumerate(state):
        if abs(amp) > atol:
            terms.append(f"({amp:.3g})|{i:0{n_qubits}b}>")
    return " + ".join(terms) if terms else "0"


def circuit_unitary(qc: Circuit) -> np.ndarray:
    """Compute the full unitary matrix of a measurement-free circuit."""
    n = qc.n_qubits
    U = np.zeros((2**n, 2**n), dtype=complex)
    for j in range(2**n):
        basis = np.zeros(2**n, dtype=complex)
        basis[j] = 1.0
        tmp = qc.copy().set_statevector(basis)
        U[:, j] = tmp.statevector()
    return U


def equal_up_to_global_phase(U: np.ndarray, V: np.ndarray, atol: float = 1e-9) -> bool:
    u = U.reshape(-1)
    v = V.reshape(-1)
    idx = None
    for k in range(len(v)):
        if abs(v[k]) > atol and abs(u[k]) > atol:
            idx = k
            break
    if idx is None:
        return np.allclose(U, V, atol=atol)
    phase = u[idx] / v[idx]
    return np.allclose(U, phase * V, atol=atol)


def permutation_unitary(n_qubits, mapping):
    """Build a unitary from a reversible basis-state mapping."""
    dim = 2**n_qubits
    U = np.zeros((dim, dim), dtype=complex)
    seen = set()
    for j in range(dim):
        k = mapping(j)
        if not (0 <= k < dim):
            raise ValueError("mapping produced an invalid basis index")
        if k in seen:
            raise ValueError("mapping is not reversible/permutational")
        seen.add(k)
        U[k, j] = 1.0
    return U


def bits_of(index, n):
    """Return basis bits in book/qcirclab order: q0 is the leftmost bit."""
    return [(index >> (n - 1 - q)) & 1 for q in range(n)]


def index_from_bits(bits):
    out = 0
    for b in bits:
        out = (out << 1) | int(b)
    return out


def bits_to_int_lsb_first(bits):
    return sum(int(b) << i for i, b in enumerate(bits))


def int_to_bits_lsb_first(value, n):
    return [(value >> i) & 1 for i in range(n)]


def set_register_int(bit_list, qubits, value, *, lsb_first=True):
    vals = int_to_bits_lsb_first(value, len(qubits)) if lsb_first else list(map(int, format(value, f"0{len(qubits)}b")))
    for q, b in zip(qubits, vals):
        bit_list[q] = b


# Subsection 4.2.1 **Principles of reversibility and information preservation**

The Toffoli gate computes a reversible AND into a target qubit. Applying it a second time uncomputes the temporary value coherently.


In [4]:
qc = Circuit(3, name="reversible-demo")

qc.ccx(0, 1, 2)  # compute
qc.ccx(0, 1, 2)  # uncompute

print(qc.draw())
print("Equivalent to identity:", np.allclose(circuit_unitary(qc), np.eye(8)))


q0: |0>─●─  ─●─
q1: |0>─●─  ─●─
q2: |0>[CCX][CCX]
Equivalent to identity: True


# Subsection 4.2.2 **Multi-controlled gates**

Multi-controlled gates generalize the Toffoli gate. They are useful for reversible logic, comparisons, arithmetic conditions, and oracle construction.


In [5]:
qc = Circuit(7, name="multi_controlled_x")
qc.mcx([0, 1, 2, 3, 4, 5], 6)

print(qc.draw())

state_in = "1111110"
qc_test = Circuit(7).initialize_basis(state_in)
qc_test.compose(qc)
print("Input: ", state_in)
print("Output:", pretty_state(qc_test.statevector()))


q0: |0>─●─
q1: |0>─●─
q2: |0>─●─
q3: |0>─●─
q4: |0>─●─
q5: |0>─●─
q6: |0>[MCX]
Input:  1111110
Output: (1+0j)|1111111>


In [6]:
qc = Circuit(11, name="nielsen_chuang_mcx")

controls = [0, 1, 2, 3, 4, 5]
target = 6
ancillas = [7, 8, 9, 10]   # m - 2 ancillas

# Compute: save partial ANDs
qc.ccx(controls[0], controls[1], ancillas[0])   # a0 = c0 AND c1
qc.ccx(ancillas[0], controls[2], ancillas[1])   # a1 = c0 c1 c2
qc.ccx(ancillas[1], controls[3], ancillas[2])   # a2 = c0 c1 c2 c3
qc.ccx(ancillas[2], controls[4], ancillas[3])   # a3 = c0 c1 c2 c3 c4

# Apply controlled-X using the last control and the accumulated AND
qc.ccx(ancillas[3], controls[5], target)

# Uncompute: restore ancillas to |0>
qc.ccx(ancillas[2], controls[4], ancillas[3])
qc.ccx(ancillas[1], controls[3], ancillas[2])
qc.ccx(ancillas[0], controls[2], ancillas[1])
qc.ccx(controls[0], controls[1], ancillas[0])

print(qc.draw())

state_in = "1111110" + "0000"   # controls, target, ancillas
qc_test = Circuit(11).initialize_basis(state_in)
qc_test.compose(qc)

print("Input: ", state_in)
print("Output:", pretty_state(qc_test.statevector()))

q0: |0> ─●─  ───  ───  ───  ───  ───  ───  ───  ─●─
q1: |0> ─●─  ───  ───  ───  ───  ───  ───  ───  ─●─
q2: |0> ───  ─●─  ───  ───  ───  ───  ───  ─●─  ───
q3: |0> ───  ───  ─●─  ───  ───  ───  ─●─  ───  ───
q4: |0> ───  ───  ───  ─●─  ───  ─●─  ───  ───  ───
q5: |0> ───  ───  ───  ───  ─●─  ───  ───  ───  ───
q6: |0> ───  ───  ───  ───  [CCX]───  ───  ───  ───
q7: |0> [CCX]─●─  ───  ───  ───  ───  ───  ─●─  [CCX]
q8: |0> ───  [CCX]─●─  ───  ───  ───  ─●─  [CCX]───
q9: |0> ───  ───  [CCX]─●─  ───  ─●─  [CCX]───  ───
q10: |0>───  ───  ───  [CCX]─●─  [CCX]───  ───  ───
Input:  11111100000
Output: (1+0j)|11111110000>


# Subsection 4.2.3 **Conventions: endianness and register layout**

In this notebook, `qcirclab` follows the book convention: qubit 0 is the top wire and the leftmost bit in basis labels.


In [7]:
qc = Circuit(3)
qc.x(0)
qc.x(1)

print("State in book/qcirclab ordering:")
print(pretty_state(qc.statevector()))
print("Qubit labels: q0 q1 q2")


State in book/qcirclab ordering:
(1+0j)|110>
Qubit labels: q0 q1 q2


# Subsection 4.3.4 **Constructing and verifying an n-bit adder**


In [17]:
# Please note that the order is not exactly the same as the
# one shown in Fig. 4.3.
def majority(qc: Circuit, a: int, b: int, c: int) -> None:
    qc.cx(c, b)
    qc.cx(c, a)
    qc.ccx(a, b, c)


def unmajority(qc: Circuit, a: int, b: int, c: int) -> None:
    qc.ccx(a, b, c)
    qc.cx(c, a)
    qc.cx(a, b)


def cuccaro_adder(n: int) -> Circuit:
    """Cuccaro ripple-carry adder.

    Layout:
        cin, a[0..n-1], b[0..n-1], cout

    Action:
        |0>|a>|b>|0> -> |0>|a>|(a+b) mod 2^n>|carry>

    The input carry cin is restored to 0.
    """
    if n <= 0:
        raise ValueError("n must be positive")

    total = 2*n + 2
    qc = Circuit(total, name=f"cuccaro_add_{n}")

    cin = 0
    a = list(range(1, 1+n))
    b = list(range(1+n, 1+2*n))
    cout = 1 + 2*n

    # Forward ripple.
    majority(qc, cin, b[0], a[0])
    for i in range(1, n):
        majority(qc, a[i-1], b[i], a[i])

    # Copy final carry.
    qc.cx(a[n-1], cout)

    # Backward uncompute.
    for i in reversed(range(1, n)):
        unmajority(qc, a[i-1], b[i], a[i])
    unmajority(qc, cin, b[0], a[0])

    return qc

In [18]:
def prepare_adder_input(n, a_val, b_val):
    qc = Circuit(2*n + 2)

    cin = 0
    a = list(range(1, 1+n))
    b = list(range(1+n, 1+2*n))
    cout = 1 + 2*n

    bits = [0] * (2*n + 2)
    bits[cin] = 0
    set_register_int(bits, a, a_val, lsb_first=True)
    set_register_int(bits, b, b_val, lsb_first=True)
    bits[cout] = 0

    return qc.initialize_basis("".join(str(x) for x in bits))


def read_adder_output(bitstring, n):
    bits = list(map(int, bitstring))

    cin = bits[0]
    a_bits = bits[1:1+n]
    b_bits = bits[1+n:1+2*n]
    cout = bits[1+2*n]

    return {
        "cin": cin,
        "a": bits_to_int_lsb_first(a_bits),
        "b": bits_to_int_lsb_first(b_bits),
        "carry_out": cout,
    }

n = 3
a_val = 3
b_val = 5

adder = cuccaro_adder(n)
qc = prepare_adder_input(n, a_val, b_val)
qc.compose(adder)

state = qc.statevector()

out = next(
    format(i, f"0{2*n+2}b")
    for i, amp in enumerate(state)
    if abs(amp) > 1e-10
)

print("Adder circuit:")
print(adder.draw())

print("\nOutput state:", pretty_state(state))
print("Bitstring:", out)
print("Decoded output:", read_adder_output(out, n))

print("Expected b:", (a_val + b_val) % (2**n),
      "carry:", (a_val + b_val) >> n)

Adder circuit:
q0: |0>────X──●─  ─────────  ─────────  ──────  ─────────  ───────●─  ─X──●─
q1: |0>─●──●─[CCX]────X──●─  ─────────  ──────  ───────●─  ─X──●─[CCX]─●────
q2: |0>─────────  ─●──●─[CCX]────X──●─  ────●─  ─X──●─[CCX]─●───────  ──────
q3: |0>─────────  ─────────  ─●──●─[CCX]─●─[CCX]─●───────  ─────────  ──────
q4: |0>─X─────●─  ─────────  ─────────  ──────  ─────────  ───────●─  ────X─
q5: |0>─────────  ─X─────●─  ─────────  ──────  ───────●─  ────X────  ──────
q6: |0>─────────  ─────────  ─X─────●─  ────●─  ────X────  ─────────  ──────
q7: |0>─────────  ─────────  ─────────  ─X────  ─────────  ─────────  ──────

Output state: (1+0j)|01100001>
Bitstring: 01100001
Decoded output: {'cin': 0, 'a': 3, 'b': 0, 'carry_out': 1}
Expected b: 0 carry: 1


Remember that 1000 (that is, carry + the 3 digits of b) is 8 in binary.

# Subsection 4.4.1 **Subtraction via additive inverses and modular wrap-around**



In [9]:
def increment_register(qc: Circuit, reg: list[int]) -> None:
    """Increment a little-endian register modulo 2^n."""
    for i in reversed(range(1, len(reg))):
        qc.mcx(reg[:i], reg[i])
    qc.x(reg[0])


def twos_complement_in_place(qc: Circuit, reg: list[int]) -> None:
    """Map b to -b mod 2^n."""
    for q in reg:
        qc.x(q)
    increment_register(qc, reg)


def cuccaro_subtractor(n: int) -> Circuit:
    """Subtraction using two's complement and Cuccaro addition.

    Layout:
        cin, a[0..n-1], b[0..n-1], cout

    Action:
        |0>|a>|b>|0> -> |0>|a>|(a-b) mod 2^n>|carry>
    """
    qc = Circuit(2*n + 2, name=f"cuccaro_sub_{n}")

    b = list(range(1+n, 1+2*n))

    # Convert b into its additive inverse modulo 2^n.
    twos_complement_in_place(qc, b)

    # Add a to (-b), leaving the result in b.
    qc.compose(cuccaro_adder(n))

    return qc


n = 3
a_val = 3
b_val = 2

subtractor = cuccaro_subtractor(n)

qc = prepare_adder_input(n, a_val, b_val)
qc.compose(subtractor)

state = qc.statevector()

out = next(
    format(i, f"0{2*n+2}b")
    for i, amp in enumerate(state)
    if abs(amp) > 1e-10
)

print("Subtractor circuit:")
print(subtractor.draw())

print("\nOutput state:", pretty_state(state))
print("Bitstring:", out)
print("Decoded output:", read_adder_output(out, n))

print("Expected b:", (a_val - b_val) % (2**n))

Subtractor circuit:
q0: |0>───  ───  ───  ───  ───  ───  ────X──●─  ─────────  ─────────  ──────  ─────────  ───────●─  ─X──●─
q1: |0>───  ───  ───  ───  ───  ───  ─●──●─[CCX]────X──●─  ─────────  ──────  ───────●─  ─X──●─[CCX]─●────
q2: |0>───  ───  ───  ───  ───  ───  ─────────  ─●──●─[CCX]────X──●─  ────●─  ─X──●─[CCX]─●───────  ──────
q3: |0>───  ───  ───  ───  ───  ───  ─────────  ─────────  ─●──●─[CCX]─●─[CCX]─●───────  ─────────  ──────
q4: |0>[ X ]───  ───  ─●─  ─●─  [ X ]─X─────●─  ─────────  ─────────  ──────  ─────────  ───────●─  ────X─
q5: |0>───  [ X ]───  ─●─  [MCX]───  ─────────  ─X─────●─  ─────────  ──────  ───────●─  ────X────  ──────
q6: |0>───  ───  [ X ][MCX]───  ───  ─────────  ─────────  ─X─────●─  ────●─  ────X────  ─────────  ──────
q7: |0>───  ───  ───  ───  ───  ───  ─────────  ─────────  ─────────  ─X────  ─────────  ─────────  ──────

Output state: (1+0j)|01101001>
Bitstring: 01101001
Decoded output: {'cin': 0, 'a': 3, 'b': 1, 'carry_out': 1}
Expected b: 1

# Subsection 4.4.2 **Comparator circuits**

An equality comparator can be built reversibly by computing bitwise XORs into one register, flipping a flag when all XOR values are zero, and then uncomputing the XORs.


In [10]:
def equality_comparator(n):
    # Layout: a[n], b[n], flag. The flag flips iff a == b.
    qc = Circuit(2*n + 1, name=f"eq_{n}")
    a = list(range(n))
    b = list(range(n, 2*n))
    flag = 2*n

    for j in range(n):
        qc.cx(a[j], b[j])

    for j in range(n):
        qc.x(b[j])
    qc.mcx(b, flag)
    for j in range(n):
        qc.x(b[j])

    for j in reversed(range(n)):
        qc.cx(a[j], b[j])
    return qc


def test_equality(n, a_val, b_val):
    bits = [0] * (2*n + 1)
    set_register_int(bits, range(n), a_val)
    set_register_int(bits, range(n, 2*n), b_val)
    qc = Circuit(2*n + 1).initialize_basis("".join(map(str, bits)))
    qc.compose(equality_comparator(n))
    return pretty_state(qc.statevector())

n = 3
print(equality_comparator(n).draw())
print("a=5, b=5 ->", test_equality(n, 5, 5))
print("a=5, b=3 ->", test_equality(n, 5, 3))


q0: |0>─●──────────  ───  ───  ───  ───  ───  ───  ───────●─
q1: |0>────●───────  ───  ───  ───  ───  ───  ───  ────●────
q2: |0>───────●────  ───  ───  ───  ───  ───  ───  ─●───────
q3: |0>─X───────[ X ]───  ───  ─●─  [ X ]───  ───  ───────X─
q4: |0>────X───────  [ X ]───  ─●─  ───  [ X ]───  ────X────
q5: |0>───────X────  ───  [ X ]─●─  ───  ───  [ X ]─X───────
q6: |0>────────────  ───  ───  [MCX]───  ───  ───  ─────────
a=5, b=5 -> (1+0j)|1011011>
a=5, b=3 -> (1+0j)|1011100>


# Subsection 4.4.3 **Conditional increment/ decrement and controlled adders**


In [11]:
n = 3
cinc = controlled_increment(n)
print(cinc.draw())

ctrl = 1
x_val = 3
bits = [0] * (n + 1)
bits[0] = ctrl
set_register_int(bits, range(1, n+1), x_val)

qc = Circuit(n + 1).initialize_basis("".join(map(str, bits)))
qc.compose(cinc)

print("Input ctrl, x =", ctrl, x_val)
print("Output:", pretty_state(qc.statevector()))
print("Expected x:", (x_val + ctrl) % (2**n))


q0: |0>─●──●─  ─●─
q1: |0>──────  [MCX]
q2: |0>───[MCX]─●─
q3: |0>─X──●─  ─●─
Input ctrl, x = 1 3
Output: (1+0j)|1101>
Expected x: 4


# Subsection 4.5.3 **Example: reversible multiplication**

In [12]:
def two_bit_multiplier() -> Circuit:
    """Two-bit reversible multiplier.

    Layout:
        a[0], a[1], b[0], b[1],
        p[0], p[1], p[2], p[3],
        t0, t1, t2, carry

    Arithmetic convention:
        a[0], b[0], and p[0] are the least significant bits.

    Action:
        |a>|b>|0000>|0000>
        ->
        |a>|b>|a*b>|0000>
    """
    qc = Circuit(12, name="mul2")

    a0, a1 = 0, 1
    b0, b1 = 2, 3

    p0, p1, p2, p3 = 4, 5, 6, 7

    t0 = 8      # stores a1*b0 temporarily
    t1 = 9      # stores a0*b1 temporarily
    t2 = 10     # stores a1*b1 temporarily
    carry = 11  # carry from the middle-bit addition

    # Least significant product bit:
    # p0 = a0*b0.
    qc.ccx(a0, b0, p0)

    # Middle partial products.
    qc.ccx(a1, b0, t0)
    qc.ccx(a0, b1, t1)

    # Add t0 and t1 into p1, with carry.
    qc.cx(t0, p1)
    qc.cx(t1, p1)
    qc.ccx(t0, t1, carry)

    # Most significant partial product.
    qc.ccx(a1, b1, t2)

    # Add t2 and carry into p2, with carry-out p3.
    qc.cx(t2, p2)
    qc.cx(carry, p2)
    qc.ccx(t2, carry, p3)

    # Uncompute temporary partial products and carry.
    qc.ccx(a1, b1, t2)
    qc.ccx(t0, t1, carry)
    qc.ccx(a0, b1, t1)
    qc.ccx(a1, b0, t0)

    return qc

In [13]:
def set_register_lsb_first(bits, qubits, value):
    for i, q in enumerate(qubits):
        bits[q] = (value >> i) & 1


def read_register_lsb_first(bits, qubits):
    return sum(bits[q] << i for i, q in enumerate(qubits))


a_val = 2
b_val = 3

a = [0, 1]
b = [2, 3]
p = [4, 5, 6, 7]

bits = [0] * 12
set_register_lsb_first(bits, a, a_val)
set_register_lsb_first(bits, b, b_val)

qc = Circuit(12).initialize_basis("".join(map(str, bits)))
qc.compose(two_bit_multiplier())

state = qc.statevector()

out = next(
    format(i, "012b")
    for i, amp in enumerate(state)
    if abs(amp) > 1e-10
)

out_bits = list(map(int, out))

print("Multiplier circuit:")
print(two_bit_multiplier().draw())

print("\nOutput state:", pretty_state(state))
print("Bitstring:", out)

print("Decoded output:")
print("a =", read_register_lsb_first(out_bits, a))
print("b =", read_register_lsb_first(out_bits, b))
print("p =", read_register_lsb_first(out_bits, p))

print("Expected product:", a_val * b_val)

Multiplier circuit:
q0: |0> ─●─  ───  ─●─  ─────────  ───  ─────────  ───  ───  ─●─  ───
q1: |0> ───  ─●─  ───  ─────────  ─●─  ─────────  ─●─  ───  ───  ─●─
q2: |0> ─●─  ─●─  ───  ─────────  ───  ─────────  ───  ───  ───  ─●─
q3: |0> ───  ───  ─●─  ─────────  ─●─  ─────────  ─●─  ───  ─●─  ───
q4: |0> [CCX]───  ───  ─────────  ───  ─────────  ───  ───  ───  ───
q5: |0> ───  ───  ───  ─X──X────  ───  ─────────  ───  ───  ───  ───
q6: |0> ───  ───  ───  ─────────  ───  ─X──X────  ───  ───  ───  ───
q7: |0> ───  ───  ───  ─────────  ───  ──────[CCX]───  ───  ───  ───
q8: |0> ───  [CCX]───  ─●─────●─  ───  ─────────  ───  ─●─  ───  [CCX]
q9: |0> ───  ───  [CCX]────●──●─  ───  ─────────  ───  ─●─  [CCX]───
q10: |0>───  ───  ───  ─────────  [CCX]─●─────●─  [CCX]───  ───  ───
q11: |0>───  ───  ───  ──────[CCX]───  ────●──●─  ───  [CCX]───  ───

Output state: (1+0j)|011101100000>
Bitstring: 011101100000
Decoded output:
a = 2
b = 3
p = 6
Expected product: 6


# Subsection 4.6.3 **Toward modular exponentiation: block architecture**

Modular exponentiation is built from controlled modular multiplications by powers of the base. The example below shows the classical control pattern that becomes a sequence of controlled reversible blocks in a quantum circuit.


In [14]:
def mod_exp_reference(base, exponent, modulus):
    """Classical repeated-squaring reference."""
    result = 1
    power = base % modulus
    e = exponent

    while e:
        if e & 1:
            result = (result * power) % modulus

        power = (power * power) % modulus
        e >>= 1

    return result


base = 2
modulus = 15

for exponent in range(8):
    print(
        f"{base}^{exponent} mod {modulus} =",
        mod_exp_reference(base, exponent, modulus)
    )

2^0 mod 15 = 1
2^1 mod 15 = 2
2^2 mod 15 = 4
2^3 mod 15 = 8
2^4 mod 15 = 1
2^5 mod 15 = 2
2^6 mod 15 = 4
2^7 mod 15 = 8


# Subsection 4.7.1 **Truth-table sampling vs. statevector checks**


In [15]:
def run_adder_once(n, a_val, b_val):
    qc = prepare_adder_input(n, a_val, b_val)
    qc.compose(cuccaro_adder(n))
    state = qc.statevector()
    out = next(
        format(i, f"0{2*n+2}b")
        for i, amp in enumerate(state)
        if abs(amp) > 1e-10
    )
    return read_adder_output(out, n)


n = 3
ok = True

for a_val in range(2**n):
    for b_val in range(2**n):
        out = run_adder_once(n, a_val, b_val)
        expected_b = (a_val + b_val) % (2**n)
        expected_carry = (a_val + b_val) >> n

        if (
            out["cin"] != 0
            or out["a"] != a_val
            or out["b"] != expected_b
            or out["carry_out"] != expected_carry
        ):
            ok = False
            print("Mismatch:", a_val, b_val, out)
            break
    if not ok:
        break

print("All Cuccaro adder basis-state tests passed:", ok)


All Cuccaro adder basis-state tests passed: True


In [16]:
n = 2
U = circuit_unitary(equality_comparator(n))
print("Comparator unitary is unitary:", np.allclose(U.conj().T @ U, np.eye(U.shape[0])))


Comparator unitary is unitary: True
